In [1]:
print("hello")

hello


In [2]:
%pwd

'c:\\Users\\shiva\\Desktop\\Shivam\\projects and programs\\projects\\medical-chatbot\\research'

In [3]:
import os 
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\shiva\\Desktop\\Shivam\\projects and programs\\projects\\medical-chatbot'

In [5]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Users\shiva\Desktop\Shivam\projects and programs\projects\medical-chatbot\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
#extract data from pdf

def load_pdf_file(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    
    documents = loader.load()
    return documents

In [7]:
extracted_data=load_pdf_file(data='Data/')

In [8]:
#extracted_data


In [21]:
#Split the Data into Text Chunks
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [22]:
text_chunks=text_split(extracted_data)
print("Length of Text Chunks", len(text_chunks))

Length of Text Chunks 5860


In [26]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [25]:
#Download the Embeddings from Hugging Face
def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings

In [27]:
embeddings=download_hugging_face_embeddings()

C:\Users\shiva\AppData\Local\Temp\ipykernel_23396\2661704553.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')


In [28]:
query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))

Length 384


In [29]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print(len(model.encode("Hello world")))


384


In [30]:
from dotenv import load_dotenv
load_dotenv()

True

In [31]:
PINECONE_API_KEY=os.environ.get('PINECONE_API_KEY')
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

In [18]:
# from pinecone.grpc import PineconeGRPC as Pinecone
# from pinecone import ServerlessSpec
# import os

# pc = Pinecone(api_key=PINECONE_API_KEY)

# index_name = "medicalbot"


# pc.create_index(
#     name=index_name,
#     dimension=384, 
#     metric="cosine", 
#     spec=ServerlessSpec(
#         cloud="aws", 
#         region="us-east-1"
#     ) 
# ) 

In [32]:
import os
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY



In [44]:
# # Embed each chunk and upsert the embeddings into your Pinecone index.
# from langchain_pinecone import PineconeVectorStore

# docsearch = PineconeVectorStore.from_documents(
#     documents=text_chunks,
#     index_name=index_name,
#     embedding=embeddings, 
# )

In [33]:
index_name = "medicalbot"

In [45]:
# from langchain_community.vectorstores import Pinecone
# docsearch = Pinecone.from_documents(
#     documents=text_chunks,
#     embedding=embeddings,
#     index_name=index_name
# )



In [ ]:
#load existing index
from langchain_community.vectorstores import Pinecone

docsearch = Pinecone.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)


In [36]:
docsearch

In [37]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [38]:
retrieved_docs = retriever.invoke("What is Acne?")


In [39]:
retrieved_docs

[Document(metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 39.0, 'page_label': '40', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data\\Medical_book.pdf', 'total_pages': 637.0}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 38.0, 'page_label': '39', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data\\Medical_book.pdf', 'total_pages': 637.0}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed. (Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 25'),
 Document(metadata={'creationdate': '

In [ ]:
# # from langchain_openai import OpenAI
# # llm = OpenAI(temperature=0.4, max_tokens=500)

# from langchain_groq import ChatGroq

# llm = ChatGroq(
#     groq_api_key=GROQ_API_KEY,   # or omit if loaded via env
#     model_name="llama3-8b-8192",
#     temperature=0.3
# )



In [40]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the context below:

    Context:
    {context}

    Question:
    {question}
    """
)

retriever = docsearch.as_retriever(search_kwargs={"k": 3})

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
)

print(chain.invoke("What is diabetes?").content)


According to the provided context, diabetes is described as a disorder of carbohydrate metabolism brought on by a combination of hereditary and environmental factors.


In [51]:
response = chain.invoke("what is skin cancer?")
print(response.content)

Skin cancer is not explicitly defined in the provided context. However, it is mentioned as one of the types of cancer that can occur in the anal area, specifically mentioning basal cell carcinomas and malignant melanomas as two types of skin cancer.


In [42]:
query = "what is skin cancer?"

docs = retriever.invoke(query)

for i, d in enumerate(docs):
    print(f"\n--- DOC {i+1} ---")
    print(d.page_content[:500])



--- DOC 1 ---
its own tissues.
Chemotherapy—The treatment of diseases, usual-
ly cancer, with drugs (chemicals).
Hair follicles—Tiny organs in the skin, each one of
which grows a single hair.
Lupus erythematosus —An autoimmune disease
that can damage skin, joints, kidneys, and other
organs.
Ringworm—A fungal infection of the skin, usually
known as tinea corporis.
Systemic—Affecting all or most parts of the body.
time, minoxidil produces satisfactory results in about one

--- DOC 2 ---
type of adenocarcinoma that can occur in the anal area
is called Paget’s disease, which can also affect the
vulva, breasts, and other areas of the body.
• Skin cancers. A small percentage of anal cancers are
either basal cell carcinomas, or malignant melanomas,
two types of skin cancer. Malignant melanomas, which
develop from skin cells that produce the brown pigment
called melanin, are far more common on areas of the
body exposed to the sun.
Approximately 3,500 Americans will be diagnosed

--- DOC 3 ---

In [43]:
print(chain.invoke("what is katu?").content)

There is no mention of "katu" in the provided context.


In [ ]:
# from pinecone import Pinecone

# pc = Pinecone(
#     api_key= "pcsk_UsQDg_BwR8t8v8KUGu8tpFGeaVszcUzu1XLqzcAM29dAMFeHtGaXKEoMz79TUv6ixMKYH"
# )

# index = pc.Index("medicalbot")

# index.delete(delete_all=True)


{}